<a href="https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ameemaiqbal/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ameemaiqbal/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship/flyrank-ml-internship


Type: Classification. My lane predicts whether a page is declining or not, a binary label (declining / not declining). This fits classification rather than clustering (I'm not grouping unlabeled pages), ranking (I'm not ordering a list, though the output can be sorted by predicted probability for prioritization), or scoring in the continuous sense (the label itself is binary, not a continuous health score).

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [9]:
print("Task type: Binary classification (declining vs. not declining)")


Task type: Binary classification (declining vs. not declining)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining, a binary label defined as whether a page's impressions dropped more than 20% month-over-month (built from trend_direction/impression comparisons, not a raw observed outcome someone manually labeled).
Source: This is a defined rule, not a directly observed outcome. It's derived from measurable data (impression trend), but the exact threshold (20%, or "down" in trend_direction) is a choice I'm making, not an objective ground truth like "this page was manually reviewed and confirmed declining." That means the label quality depends entirely on whether that threshold reasonably captures what a content team would actually call "worth reviewing", it's a proxy for the real judgment call, not the judgment call itself.

In [10]:
# Target/proxy framing — no computation needed, reasoning is in the text cell above.
print("Target: is_declining (proxy label, rule-based on impression trend, not a human-verified outcome)")

Target: is_declining (proxy label, rule-based on impression trend, not a human-verified outcome)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@50 (used throughout Notebooks 01–03). Of the top 50 pages my model flags as "review first," what fraction are actually declining? I'm choosing this over plain accuracy because the decision this supports is prioritization under limited review capacity, a content team can only manually check so many pages per week, so what matters is whether the top of the ranked list is trustworthy, not whether the model is right on average across all 30,000+ pages. A "good" result is one that clearly beats the baseline hand-written rule (0.24 in Notebook 01) and ideally beats the naive majority-class base rate too, both of which I've already measured in earlier notebooks (baseline rule ≈0.24, model up to ≈0.74 in the starter pipeline; my own position_volatility model reached 0.732 accuracy with GroupShuffleSplit validation).

In [11]:
# Success metric framing — no computation needed, reasoning is in the text cell above.
print("Success metric: Precision@50 (top-of-list trustworthiness for limited review capacity)")


Success metric: Precision@50 (top-of-list trustworthiness for limited review capacity)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one page (content item), at a single snapshot in time, with its own impression trend, position, CTR, and metadata. Each row represents a distinct piece of content that the model would independently make a "declining or not" prediction for.

In [12]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows (pages): {len(df):,}")
print(f"Columns: {list(df.columns)}")
df[["content_type", "avg_position", "impressions_90d", "ctr", "word_count", "trend_direction", "is_declining"]].head(5)

Rows (pages): 30,000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining']


,content_type,avg_position,impressions_90d,ctr,word_count,trend_direction,is_declining
0,keyword article,10.6,3803,0.76,3221.0,down,1
1,keyword article,20.3,15320,0.05,2481.0,down,1
2,keyword article,36.5,12581,0.09,3515.0,down,1
3,keyword article,6.2,11751,0.49,NaN,stable,0
4,keyword article,44.0,19140,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule (e.g. "flag pages with position > X and impressions dropping") can't capture how these signals interact differently across content types, clients, and traffic levels. My Week 3 finding showed position_volatility alone was more predictive than raw impression trend, but that relationship isn't a clean threshold, a page can be volatile for many different underlying reasons (seasonal demand, a competitor change, an algorithm update), and the combination of volatility with impression level and content age is what separates a temporary dip from a genuine decline. A fixed if/else rule would need dozens of hand-tuned thresholds to approximate what a model learns automatically from the data's actual patterns, and it would still miss interactions between features that a model like a random forest can pick up without being explicitly told to look for them.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.